In [10]:
%%writefile HPC_2B.cpp
#include <iostream>
#include <omp.h>
using namespace std;

void merge(int a[], int l, int m, int r) {
    int temp[100], i=l, j=m+1, k=0;

    while(i<=m && j<=r)
        temp[k++] = (a[i]<a[j]) ? a[i++] : a[j++];

    while(i<=m) temp[k++] = a[i++];
    while(j<=r) temp[k++] = a[j++];

    for(i=l, k=0; i<=r; i++, k++)
        a[i] = temp[k];
}

// Sequential
void seqMergeSort(int a[], int l, int r) {
    if(l < r) {
        int m = (l+r)/2;
        seqMergeSort(a, l, m);
        seqMergeSort(a, m+1, r);
        merge(a, l, m, r);
    }
}

// Parallel
void parMergeSort(int a[], int l, int r) {
    if(l < r) {
        int m = (l+r)/2;

        #pragma omp parallel sections
        {
            #pragma omp section
            parMergeSort(a, l, m);

            #pragma omp section
            parMergeSort(a, m+1, r);
        }

        merge(a, l, m, r);
    }
}

int main() {
    int n;
    cin >> n;

    int a[100], b[100];
    for(int i=0;i<n;i++) {
        cin >> a[i];
        b[i] = a[i];
    }

    // Sequential time
    double start = omp_get_wtime();
    seqMergeSort(a, 0, n-1);
    double end = omp_get_wtime();
    cout << "Sequential Time: " << end-start << endl;

    // Parallel time
    start = omp_get_wtime();
    parMergeSort(b, 0, n-1);
    end = omp_get_wtime();
    cout << "Parallel Time: " << end-start << endl;

    cout << "Sorted Array:\n";
    for(int i=0;i<n;i++)
        cout << b[i] << " ";

    return 0;
}


Overwriting HPC_2B.cpp


In [11]:
!g++ -fopenmp HPC_2B.cpp -o HPC_2B

In [12]:
!./HPC_2B

5
60
8
40
2
0
Sequential Time: 1.314e-06
Parallel Time: 0.00395521
Sorted Array:
0 2 8 40 60 

## Detailed Viva Notes: HPC_2B (Sequential vs Parallel Merge Sort)

This notebook compares **sequential merge sort** and **parallel merge sort** using OpenMP sections.

### Program 1: Write C++ source (`HPC_2B.cpp`)

- `merge(...)`: merges two sorted halves into one sorted segment.
- `seqMergeSort(...)`: recursive divide-and-conquer merge sort.
- `parMergeSort(...)`: recursive merge sort with:
  - `#pragma omp parallel sections`
  - left and right halves sorted concurrently.
- Main:
  - Read `n` and input values
  - Copy into `a` and `b`
  - Measure sequential and parallel times
  - Print sorted array.

**Example input**

- `n = 8`
- `9 1 6 3 7 2 8 4`

**Expected output (example)**

- `Sequential Time: ...`
- `Parallel Time: ...`
- `Sorted Array:`
  - `1 2 3 4 6 7 8 9`

**Important viva point**

- `temp[100]` and arrays `a[100], b[100]` imply max safe input around 100 elements.

### Program 2: Compile with OpenMP

- `g++ -fopenmp HPC_2B.cpp -o HPC_2B`
- Expected output: successful compilation.

### Program 3: Run executable

- Produces timings and sorted result.
- Output values should always be sorted ascending.

---

## Viva Quick Answers

1. Why merge sort suits parallelism? Left/right subproblems are independent.
2. Why use sections pragma? To run disjoint recursive branches concurrently.
3. Why sorted output validates correctness? Merge sort invariants enforce global ordering after merges.
